# Training Dataset Download

Check for local folder `BraTS2020_TrainingData` and use it if present; otherwise download `awsaf49/brats20-dataset-training-validation` from Kaggle, copying `kaggle.json` to `~/.kaggle` if available.

In [ ]:
import kagglehub
import shutil

# SMART SETUP
# Expected local dataset folder
local_folder = 'BraTS2020_TrainingData'

# Use local copy if available; otherwise attempt to download from Kaggle
if os.path.exists(local_folder):
    print(f"Found local dataset at '{local_folder}'. Using local files.")
    path_to_use = local_folder
else:
    print(f"Local dataset folder '{local_folder}' not found. Initiating Kaggle download.")
    try:
        # If a local kaggle.json credential file exists, copy it to ~/.kaggle for authentication
        if os.path.exists('kaggle.json'):
            dest = os.path.expanduser('~/.kaggle')
            os.makedirs(dest, exist_ok=True)
            shutil.copy('kaggle.json', os.path.join(dest, 'kaggle.json'))

        # Download dataset and capture the returned path
        path_to_use = kagglehub.dataset_download("awsaf49/brats20-dataset-training-validation")
        print(f"Download completed. Dataset path: {path_to_use}")

    except Exception:
        print("Download failed. Please ensure 'kaggle.json' is present and valid.")
        raise

# Dataset Preprocessing

This section configures the environment and identifies relevant MRI files.

**Key Parameters**
- `DATASET_PATH`: Directory containing the BraTS dataset.
- `CHUNK_SIZE = 2000`: Processed data are saved in batches of 2000 slices to optimize memory usage.

**File Discovery**
The script recursively searches for `.nii` files containing 'flair' in their names, which serve as reference images for subsequent processing and mask alignment.


In [ ]:
import os

# Set dataset and output paths
DATASET_PATH = 'BraTS2020_TrainingData'
OUTPUT_PATH = 'processed_tumor_packs'
CHUNK_SIZE = 2000  # Number of slices per output file

os.makedirs(OUTPUT_PATH, exist_ok=True)
print(f"Looking for data in: {DATASET_PATH}...")

# Locate FLAIR .nii files in the dataset directory
files_found = []
for root, dirs, files in os.walk(DATASET_PATH):
    for file in files:
        if file.endswith('.nii') and 'flair' in file.lower():
            files_found.append(os.path.join(root, file))

print(f"Found {len(files_found)} patients. Starting processing...")

# Data Normalization and Packing

This cell implements the patient-wise preprocessing pipeline used to prepare FLAIR volumes and corresponding segmentation masks for downstream modeling.

## Pipeline overview
- Load volumes: read `FLAIR` (image) and `SEG` (tumor mask) using `nibabel`.
- Global intensity normalization: rescale intensities to the range [-1, 1] using
    \[
    x_{\mathrm{norm}} = 2\cdot\frac{x - x_{\min}}{x_{\max} - x_{\min}} - 1,
    \]
    applied per FLAIR volume. Volumes with zero dynamic range are skipped.
- Slice extraction and filtering:
    - Convert the 3D volumes into 2D axial slices.
    - Retain only slices for which the segmentation mask contains tumor tissue (`np.max(seg_slice) > 0`).
- Packing and storage:
    - Convert retained image and mask slices to `float16` to reduce storage.
    - Stack image and mask as a two-channel tensor (channel 0 = image, channel 1 = mask).
    - Accumulate slices into fixed-size packs (`CHUNK_SIZE`, default 2000). When a pack reaches `CHUNK_SIZE`, save it as a `.pt` file and clear the buffer.
    - Save any remaining slices as a final pack.

### Notes
- The procedure prioritizes storage efficiency and removes non-informative background slices to bias the dataset toward tumor-containing examples.
- Ensure `DATASET_PATH`, `OUTPUT_PATH`, and `CHUNK_SIZE` are set appropriately before execution.

In [ ]:
from tqdm import tqdm
import numpy as np
import nibabel as nib
import torch
import os

# Processing
current_chunk = []
pack_counter = 0
total_saved = 0

for flair_path in tqdm(files_found, desc="Processing patients"):
    try:
        # find matching segmentation
        seg_path = flair_path.replace('flair', 'seg').replace('FLAIR', 'seg')
        if not os.path.exists(seg_path):
            continue

        # load volumes
        vol_flair = nib.load(flair_path).get_fdata()
        vol_seg = nib.load(seg_path).get_fdata()

        # global intensity normalization to [-1, 1]
        min_v, max_v = np.min(vol_flair), np.max(vol_flair)
        if max_v - min_v <= 0:
            continue
        vol_flair = 2 * ((vol_flair - min_v) / (max_v - min_v)) - 1

        # slice-wise filtering and packing (retain slices with tumor)
        for i in range(vol_flair.shape[2]):
            seg_slice = vol_seg[:, :, i]
            if np.max(seg_slice) <= 0:
                continue

            flair_slice = vol_flair[:, :, i]

            # convert to float16 and form 2-channel tensor [2, H, W]
            t_img = torch.from_numpy(flair_slice.astype(np.float16)).unsqueeze(0)
            t_seg = torch.from_numpy(seg_slice.astype(np.float16)).unsqueeze(0)
            combined = torch.cat([t_img, t_seg], dim=0)

            current_chunk.append(combined)

            # save full pack
            if len(current_chunk) >= CHUNK_SIZE:
                save_name = f"tumor_pack_{pack_counter}.pt"
                save_path = os.path.join(OUTPUT_PATH, save_name)
                torch.save(torch.stack(current_chunk), save_path)
                print(f"Saved {save_name} ({CHUNK_SIZE} slices)")
                current_chunk = []
                pack_counter += 1
                total_saved += CHUNK_SIZE

    except Exception as e:
        print(f"Error processing {flair_path}: {e}")

# save remainder
final_pack_added = 0
if current_chunk:
    save_name = f"tumor_pack_{pack_counter}.pt"
    save_path = os.path.join(OUTPUT_PATH, save_name)
    torch.save(torch.stack(current_chunk), save_path)
    saved_now = len(current_chunk)
    total_saved += saved_now
    final_pack_added = 1
    print(f"Saved {save_name} ({saved_now} slices)")

total_packs = pack_counter + final_pack_added
print(f"Completed. Total slices: {total_saved}. Packs: {total_packs}. Output: {OUTPUT_PATH}")


# Sanity Check

Following the preprocessing phase, this block performs a diagnostic check on the serialized `.pt` files to ensure data integrity and pipeline reliability.

### **Verification Metrics**
1.  **Tensor Dimensionality:**
    * Expected dimensions: `[N, 2, 240, 240]`.
    * **Channel 0:** Normalized MRI Image.
    * **Channel 1:** Segmentation Mask.
2.  **Normalization Bounds:**
    * Verifies that pixel intensities strictly adhere to the **`[-1, 1]`** interval required by the model architecture.
3.  **Spatial Alignment:**
    * A random slice is visualized to confirm that the segmentation mask strictly overlaps with the tumor region in the MRI.
    * *Note:* Misalignment at this stage indicates a critical failure in the data loading logic.

In [ ]:
import os
import random
import torch
import matplotlib.pyplot as plt

# Configuration
INPUT_FOLDER = 'processed_tumor_packs'

# Locate processed .pt files
pack_files = sorted([f for f in os.listdir(INPUT_FOLDER) if f.endswith('.pt')])

if not pack_files:
    print(f"No .pt files found in '{INPUT_FOLDER}'. Ensure the preprocessing step completed successfully.")
else:
    print(f"Found {len(pack_files)} pack(s). Inspecting the first pack: '{pack_files[0]}'")

    # Load the first pack onto CPU for inspection
    pack_path = os.path.join(INPUT_FOLDER, pack_files[0])
    data_pack = torch.load(pack_path, map_location='cpu')

    # Tensor statistics
    print("\nTensor statistics")
    print(f"Shape:       {data_pack.shape}  (Batch, Channels, Height, Width)")
    print(f"Dtype:       {data_pack.dtype}")
    print(f"Memory:      {data_pack.element_size() * data_pack.numel() / 1e6:.2f} MB")

    # Image and mask channels
    img_channel = data_pack[:, 0, :, :]
    mask_channel = data_pack[:, 1, :, :]

    print(f"Image range: min {img_channel.min():.4f}, max {img_channel.max():.4f}")
    print(f"Mask range:  min {mask_channel.min():.4f}, max {mask_channel.max():.4f}")
    print(f"Tumor ratio: {(mask_channel > 0).float().mean().item() * 100:.2f}% of pixels in this pack are tumor")

    # Visual sanity check (random slice)
    idx = random.randrange(len(data_pack))
    sample = data_pack[idx]
    img = sample[0].float()
    mask = sample[1].float()

    fig, ax = plt.subplots(1, 3, figsize=(15, 5))
    ax[0].imshow(img, cmap='gray')
    ax[0].set_title("MRI Slice (normalized)")
    ax[0].axis('off')

    ax[1].imshow(mask, cmap='hot')
    ax[1].set_title("Ground Truth Segmentation")
    ax[1].axis('off')

    ax[2].imshow(img, cmap='gray')
    ax[2].imshow(mask, cmap='hot', alpha=0.4)
    ax[2].set_title("Overlay")
    ax[2].axis('off')

    plt.tight_layout()
    plt.show()


# Autoencoder Architecture

This section defines the autoencoder and discriminator architecture, configures loss functions, and prepares the training pipeline.

## Components
- **AutoencoderKL:** Variational autoencoder for learning latent representations of tumor MRI slices.
- **PatchDiscriminator:** Adversarial discriminator to enforce realistic image reconstruction.
- **Loss Functions:**
    - Perceptual loss: Ensures high-level feature consistency.
    - Adversarial loss: Encourages realistic synthetic samples.
    - Reconstruction loss: Minimizes pixel-level differences.

## Configuration
- Device: GPU (CUDA) if available, otherwise CPU.
- Batch size and learning rate optimized for BraTS dataset.
- Training loop includes periodic validation and checkpoint saving.

In [ ]:
# Install required packages in the notebook kernel (use %pip so installations affect the running environment)

# Configure deterministic behavior for reproducibility
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from monai.utils import set_determinism

# Import generative model components and loss functions
from generative.networks.nets import AutoencoderKL, PatchDiscriminator
from generative.losses import PatchAdversarialLoss, PerceptualLoss

from tqdm import tqdm
# Conditional import for Google Colab drive integration (available only in Colab)
try:
    from google.colab import drive
except ImportError:
    drive = None

import matplotlib.pyplot as plt

# Enforce deterministic behavior for reproducible experiments
set_determinism(42)

print("Package installation completed and modules imported successfully.")


# MRI Tumor Slice Autoencoder Training

This notebook trains a 2D variational autoencoder with adversarial regularization
on randomly sampled MRI slices stored as `.pt` tensors.

**Architecture**
- AutoencoderKL (MONAI Generative)
- PatchGAN discriminator

**Training Features**
- Mixed precision (AMP)
- KL regularization
- Patch adversarial loss
- Random slice sampling from volumetric packs
